# C03–M2–L1 — Similarity Search on Employee Records using Python and Chroma DB (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original files](https://github.com/hengkuankian/learning_ibm_rag_agentic/tree/main/Course3_Vector_Databases_for_RAG/Similarity_Search_on_Employee_Records_using_Python_and_Chroma_DB) · cross-checked with the [Hugging Face reference port](https://github.com/oooookk7/huggingface-rag-and-agentic-ai/tree/main/03-vector-databases-for-rag)

**Before you run:** add **`GOOGLE_API_KEY`** to Colab 🔑 Secrets (free key from aistudio.google.com/apikey). CPU runtime is enough — no GPU, no model downloads.

**What changed vs. the IBM version (CP-G.3 Gemini migration)**
- Setup: Cell 1 = minimal pip install (+ `chromadb`), Cell 2 = universal Gemini init. Legacy Groq/HF setup, `get_llm()` and watsonx shim removed.
- New `%%writefile gemini_ef.py` (`GeminiEmbeddingFunction`). `data.py`, `similarity_employeedata.py` and `books_advanced_search.py` recreated with `%%writefile` in the original layout; only change: `SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")` → `GeminiEmbeddingFunction()`.
- Colab exercises: query the collection in the notebook, metadata + `where_document` filters, LLM staffing assistant with `llm.invoke(...).text`.

Everything else — explanations, code, exercises — is the original lab.

In [ ]:
# Minimal fast install (~5-10 seconds)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai chromadb

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

> 💡 **How to use the ✅ Answer cells:** every exercise is followed by a hidden, collapsed cell titled *✅ Answer — …*. Try the exercise yourself first. To see the reference answer, double-click the cell (or *Show code*), tick **run_answer** and run it. While the box is unticked the cell raises a short *Answer not run* message on purpose, so **Runtime ▸ Run all** stops right before the first answer instead of spoiling it — after that, run the remaining cells one by one (or tick the boxes you want and use *Run after*).

### `gemini_ef.py` — Gemini embedding function for Chroma
A tiny `chromadb.EmbeddingFunction` over `GoogleGenerativeAIEmbeddings` (swap rule: *Raw ChromaDB → GeminiEmbeddingFunction*). The lab scripts import it instead of `SentenceTransformerEmbeddingFunction`; they read `GOOGLE_API_KEY` from the environment set in Cell 2.

In [ ]:
%%writefile gemini_ef.py
# Chroma embedding function backed by Gemini (replaces SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2"))
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
from chromadb import Documents, EmbeddingFunction, Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model: str = "models/gemini-embedding-2"):
        self.model = model
        self._emb = GoogleGenerativeAIEmbeddings(model=model)   # key read from GOOGLE_API_KEY

    def __call__(self, input: Documents) -> Embeddings:
        texts = list(input)
        out = []
        for i in range(0, len(texts), 100):              # free tier: <= 100 texts per call
            out += self._emb.embed_documents(texts[i:i + 100])
        return out

    @staticmethod
    def name() -> str:
        return "gemini"

    def get_config(self):
        return {"model": self.model}

    @staticmethod
    def build_from_config(config):
        return GeminiEmbeddingFunction(config.get("model", "models/gemini-embedding-2"))

## Project files
`data.py` holds the lab data: `EMPLOYEES` (15 records with name, role, department, experience, skills, location, employment type) and `BOOKS` for the practice exercise.

In [ ]:
%%writefile data.py
# Defining a list of employee dictionaries
# Each dictionary represents an individual employee with comprehensive information
EMPLOYEES = [
    {
        "id": "employee_1",
        "name": "John Doe",
        "experience": 5,
        "department": "Engineering",
        "role": "Software Engineer",
        "skills": "Python, JavaScript, React, Node.js, databases",
        "location": "New York",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_2",
        "name": "Jane Smith",
        "experience": 8,
        "department": "Marketing",
        "role": "Marketing Manager",
        "skills": "Digital marketing, SEO, content strategy, analytics, social media",
        "location": "Los Angeles",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_3",
        "name": "Alice Johnson",
        "experience": 3,
        "department": "HR",
        "role": "HR Coordinator",
        "skills": "Recruitment, employee relations, HR policies, training programs",
        "location": "Chicago",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_4",
        "name": "Michael Brown",
        "experience": 12,
        "department": "Engineering",
        "role": "Senior Software Engineer",
        "skills": "Java, Spring Boot, microservices, cloud architecture, DevOps",
        "location": "San Francisco",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_5",
        "name": "Emily Wilson",
        "experience": 2,
        "department": "Marketing",
        "role": "Marketing Assistant",
        "skills": "Content creation, email marketing, market research, social media management",
        "location": "Austin",
        "employment_type": "Part-time"
    },
    {
        "id": "employee_6",
        "name": "David Lee",
        "experience": 15,
        "department": "Engineering",
        "role": "Engineering Manager",
        "skills": "Team leadership, project management, software architecture, mentoring",
        "location": "Seattle",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_7",
        "name": "Sarah Clark",
        "experience": 8,
        "department": "HR",
        "role": "HR Manager",
        "skills": "Performance management, compensation planning, policy development, conflict resolution",
        "location": "Boston",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_8",
        "name": "Chris Evans",
        "experience": 20,
        "department": "Engineering",
        "role": "Senior Architect",
        "skills": "System design, distributed systems, cloud platforms, technical strategy",
        "location": "New York",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_9",
        "name": "Jessica Taylor",
        "experience": 4,
        "department": "Marketing",
        "role": "Marketing Specialist",
        "skills": "Brand management, advertising campaigns, customer analytics, creative strategy",
        "location": "Miami",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_10",
        "name": "Alex Rodriguez",
        "experience": 18,
        "department": "Engineering",
        "role": "Lead Software Engineer",
        "skills": "Full-stack development, React, Python, machine learning, data science",
        "location": "Denver",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_11",
        "name": "Hannah White",
        "experience": 6,
        "department": "HR",
        "role": "HR Business Partner",
        "skills": "Strategic HR, organizational development, change management, employee engagement",
        "location": "Portland",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_12",
        "name": "Kevin Martinez",
        "experience": 10,
        "department": "Engineering",
        "role": "DevOps Engineer",
        "skills": "Docker, Kubernetes, AWS, CI/CD pipelines, infrastructure automation",
        "location": "Phoenix",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_13",
        "name": "Rachel Brown",
        "experience": 7,
        "department": "Marketing",
        "role": "Marketing Director",
        "skills": "Strategic marketing, team leadership, budget management, campaign optimization",
        "location": "Atlanta",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_14",
        "name": "Matthew Garcia",
        "experience": 3,
        "department": "Engineering",
        "role": "Junior Software Engineer",
        "skills": "JavaScript, HTML/CSS, basic backend development, learning frameworks",
        "location": "Dallas",
        "employment_type": "Full-time"
    },
    {
        "id": "employee_15",
        "name": "Olivia Moore",
        "experience": 12,
        "department": "Engineering",
        "role": "Principal Engineer",
        "skills": "Technical leadership, system architecture, performance optimization, mentoring",
        "location": "San Francisco",
        "employment_type": "Full-time"
    },
]

# List of book dictionaries with comprehensive details for advanced search
BOOKS = [
    {
        "id": "book_1",
        "title": "The Great Gatsby",
        "author": "F. Scott Fitzgerald",
        "genre": "Classic",
        "year": 1925,
        "rating": 4.1,
        "pages": 180,
        "description": "A tragic tale of wealth, love, and the American Dream in the Jazz Age",
        "themes": "wealth, corruption, American Dream, social class",
        "setting": "New York, 1920s"
    },
    {
        "id": "book_2",
        "title": "To Kill a Mockingbird",
        "author": "Harper Lee",
        "genre": "Classic",
        "year": 1960,
        "rating": 4.3,
        "pages": 376,
        "description": "A powerful story of racial injustice and moral growth in the American South",
        "themes": "racism, justice, moral courage, childhood innocence",
        "setting": "Alabama, 1930s"
    },
    {
        "id": "book_3",
        "title": "1984",
        "author": "George Orwell",
        "genre": "Dystopian",
        "year": 1949,
        "rating": 4.4,
        "pages": 328,
        "description": "A chilling vision of totalitarian control and surveillance society",
        "themes": "totalitarianism, surveillance, freedom, truth",
        "setting": "Oceania, dystopian future"
    },
    {
        "id": "book_4",
        "title": "Harry Potter and the Philosopher's Stone",
        "author": "J.K. Rowling",
        "genre": "Fantasy",
        "year": 1997,
        "rating": 4.5,
        "pages": 223,
        "description": "A young wizard discovers his magical heritage and begins his education at Hogwarts",
        "themes": "friendship, courage, good vs evil, coming of age",
        "setting": "England, magical world"
    },
    {
        "id": "book_5",
        "title": "The Lord of the Rings",
        "author": "J.R.R. Tolkien",
        "genre": "Fantasy",
        "year": 1954,
        "rating": 4.5,
        "pages": 1216,
        "description": "An epic fantasy quest to destroy a powerful ring and save Middle-earth",
        "themes": "heroism, friendship, good vs evil, power corruption",
        "setting": "Middle-earth, fantasy realm"
    },
    {
        "id": "book_6",
        "title": "The Hitchhiker's Guide to the Galaxy",
        "author": "Douglas Adams",
        "genre": "Science Fiction",
        "year": 1979,
        "rating": 4.2,
        "pages": 224,
        "description": "A humorous space adventure following Arthur Dent across the galaxy",
        "themes": "absurdity, technology, existence, humor",
        "setting": "Space, various planets"
    },
    {
        "id": "book_7",
        "title": "Dune",
        "author": "Frank Herbert",
        "genre": "Science Fiction",
        "year": 1965,
        "rating": 4.3,
        "pages": 688,
        "description": "A complex tale of politics, religion, and ecology on a desert planet",
        "themes": "power, ecology, religion, politics",
        "setting": "Arrakis, distant future"
    },
    {
        "id": "book_8",
        "title": "The Hunger Games",
        "author": "Suzanne Collins",
        "genre": "Dystopian",
        "year": 2008,
        "rating": 4.2,
        "pages": 374,
        "description": "A teenage girl fights for survival in a brutal televised competition",
        "themes": "survival, oppression, sacrifice, rebellion",
        "setting": "Panem, dystopian future"
    },
]

## Part 1 — Employee similarity search
Creates `employee_collection` (cosine HNSW space), turns each employee into a text document, adds it with metadata, then `perform_advanced_search()` runs similarity searches (Python developers, leadership roles), metadata filters (`$eq`, `$gte`, `$in`) and combined query + filter searches.

In [ ]:
%%writefile similarity_employeedata.py
# Importing necessary modules from the chromadb package:
# chromadb is used to interact with the Chroma DB database,
# embedding_functions is used to define the embedding model
import chromadb
from chromadb.utils import embedding_functions
from gemini_ef import GeminiEmbeddingFunction  # Colab: Gemini embeddings
from data import EMPLOYEES

# Define the embedding function (Colab: Gemini instead of SentenceTransformers)
# This function will be used to generate embeddings (vector representations) for the data
ef = GeminiEmbeddingFunction()

# Creating an instance of ChromaClient to establish a connection with the Chroma database
client = chromadb.Client()

# Defining a name for the collection where data will be stored or accessed
# This collection is likely used to group related records, such as employee data
collection_name = "employee_collection"

#Perform various types of searches within the collection
def perform_advanced_search(collection, all_items):
    try:
        # Advanced search operations will be placed here
        print("=== Similarity Search Examples ===")

        # Example 1: Search for Python developers
        print("\n1. Searching for Python developers:")
        query_text = "Python developer with web development experience"
        results = collection.query(
            query_texts=[query_text],
            n_results=3
        )
        print(f"Query: '{query_text}'")
        for i, (doc_id, document, distance) in enumerate(zip(
            results['ids'][0], results['documents'][0], results['distances'][0]
        )):
            metadata = results['metadatas'][0][i]
            print(f"  {i+1}. {metadata['name']} ({doc_id}) - Distance: {distance:.4f}")
            print(f"     Role: {metadata['role']}, Department: {metadata['department']}")
            print(f"     Document: {document[:100]}...")

        # Example 2: Search for leadership roles
        print("\n2. Searching for leadership and management roles:")
        query_text = "team leader manager with experience"
        results = collection.query(
            query_texts=[query_text],
            n_results=3
        )
        print(f"Query: '{query_text}'")
        for i, (doc_id, document, distance) in enumerate(zip(
            results['ids'][0], results['documents'][0], results['distances'][0]
        )):
            metadata = results['metadatas'][0][i]
            print(f"  {i+1}. {metadata['name']} ({doc_id}) - Distance: {distance:.4f}")
            print(f"     Role: {metadata['role']}, Experience: {metadata['experience']} years")
    
        print("\n=== Metadata Filtering Examples ===")

        # Example 1: Filter by department
        print("\n3. Finding all Engineering employees:")
        results = collection.get(
            where={"department": "Engineering"}
        )
        print(f"Found {len(results['ids'])} Engineering employees:")
        for i, doc_id in enumerate(results['ids']):
            metadata = results['metadatas'][i]
            print(f"  - {metadata['name']}: {metadata['role']} ({metadata['experience']} years)")

        # Example 2: Filter by experience range
        print("\n4. Finding employees with 10+ years experience:")
        results = collection.get(
            where={"experience": {"$gte": 10}}
        )
        print(f"Found {len(results['ids'])} senior employees:")
        for i, doc_id in enumerate(results['ids']):
            metadata = results['metadatas'][i]
            print(f"  - {metadata['name']}: {metadata['role']} ({metadata['experience']} years)")

        # Example 3: Filter by location
        print("\n5. Finding employees in California:")
        results = collection.get(
            where={"location": {"$in": ["San Francisco", "Los Angeles"]}}
        )
        print(f"Found {len(results['ids'])} employees in California:")
        for i, doc_id in enumerate(results['ids']):
            metadata = results['metadatas'][i]
            print(f"  - {metadata['name']}: {metadata['location']}")

        print("\n=== Combined Search: Similarity + Metadata Filtering ===")

        # Example: Find experienced Python developers in specific locations
        print("\n6. Finding senior Python developers in major tech cities:")
        query_text = "senior Python developer full-stack"
        results = collection.query(
            query_texts=[query_text],
            n_results=5,
            where={
                "$and": [
                    {"experience": {"$gte": 8}},
                    {"location": {"$in": ["San Francisco", "New York", "Seattle"]}}
                ]
            }
        )
        print(f"Query: '{query_text}' with filters (8+ years, major tech cities)")
        print(f"Found {len(results['ids'][0])} matching employees:")
        for i, (doc_id, document, distance) in enumerate(zip(
            results['ids'][0], results['documents'][0], results['distances'][0]
        )):
            metadata = results['metadatas'][0][i]
            print(f"  {i+1}. {metadata['name']} ({doc_id}) - Distance: {distance:.4f}")
            print(f"     {metadata['role']} in {metadata['location']} ({metadata['experience']} years)")
            print(f"     Document snippet: {document[:80]}...")
   
        # Check if the results are empty or undefined
        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            # Log a message if no similar documents are found for the query term
            print(f'No documents found similar to "{query_text}"')
            return

        # Log the header for the top 3 similar documents based on the query term
        print(f'Top 3 similar documents to "{query_text}":')
        # Loop through the top 3 results and log the document details
        for i in range(min(3, len(results['ids'][0]))):
            # Extract the document ID and similarity score from the results
            doc_id = results['ids'][0][i]
            score = results['distances'][0][i]
            # Retrieve the document text corresponding to the current ID from the results
            text = results['documents'][0][i]
            # Check if the text is available; if not, log 'Text not available'
            if not text:
                print(f' - ID: {doc_id}, Text: "Text not available", Score: {score:.4f}')
            else:
                print(f' - ID: {doc_id}, Text: "{text}", Score: {score:.4f}')

    except Exception as error:
        print(f"Error in advanced search: {error}")

# Defining a function named 'main'
# This function is used to encapsulate the main operations for creating collections,
# generating embeddings, and performing similarity search
def main():
    try:
        # Code for database operations will be placed here
        # This includes creating collections, adding data, and performing searches
        # Creating a collection using the ChromaClient instance
        # The 'create_collection' method creates a new collection with the specified configuration
        collection = client.create_collection(
            # Specifying the name of the collection to be created
            name=collection_name,
            # Adding metadata to describe the collection
            metadata={"description": "A collection for storing employee data"},
            # Configuring the collection with cosine distance and embedding function
            configuration={
                "hnsw": {"space": "cosine"},
                "embedding_function": ef
            }
        )

        employees = EMPLOYEES
        # Create comprehensive text documents for each employee
        # These documents will be used for similarity search based on skills, roles, and experience
        employee_documents = []
        for employee in employees:
            document = f"{employee['role']} with {employee['experience']} years of experience in {employee['department']}. "
            document += f"Skills: {employee['skills']}. Located in {employee['location']}. "
            document += f"Employment type: {employee['employment_type']}."
            employee_documents.append(document)

        # Adding data to the collection in the Chroma database
        # The 'add' method inserts or updates data into the specified collection
        collection.add(
            # Extracting employee IDs to be used as unique identifiers for each record
            ids=[employee["id"] for employee in employees],
            # Using the comprehensive text documents we created
            documents=employee_documents,
            # Adding comprehensive metadata for filtering and search
            metadatas=[{
                "name": employee["name"],
                "department": employee["department"],
                "role": employee["role"],
                "experience": employee["experience"],
                "location": employee["location"],
                "employment_type": employee["employment_type"]
            } for employee in employees]
        )

        print(f"Collection created: {collection.name}")



    except Exception as error:
        # Catching and handling any errors that occur within the 'try' block
        # Logs the error message to the console for debugging purposes
        print(f"Error: {error}")

    # Retrieving all items from the specified collection
    # The 'get' method fetches all records stored in the collection
    all_items = collection.get()
    # Logging the retrieved items to the console for inspection or debugging
    print("Collection contents:")
    print(f"Number of documents: {len(all_items['documents'])}")

    # Call the perform_advanced_search function with the collection and all_items as arguments
    perform_advanced_search(collection, all_items)

if __name__ == '__main__':
    main()

In [ ]:
!python similarity_employeedata.py

## Part 2 — Practice exercise: book recommendations
`BOOKS` (8 books, in `data.py`) is the lab's book list. The lab's 4 exercises + Bonus are below; each has a task, an empty cell for your code and a hidden ✅ Answer. Exercises 1–3 can be done in the notebook first; Exercise 4 puts them into `books_advanced_search.py` and runs it.

> ⚠️ The script this notebook used to ship (copied from a GitHub student solution) filtered only `"Fantasy"` for *Fantasy or Science Fiction* (found 2 of 4 books), printed *"Searching for Python developers"* for the book search and had no genre filter in the dystopian search. The Exercise 4 answer contains the fixed script.

### Exercise 1 · Implement similarity search for book recommendations
Create meaningful text documents for each book that combine title, description, themes and setting information for semantic search.

*Hint: combine multiple book attributes into descriptive text documents.*

In [ ]:
# Your code here

In [ ]:
# @title ✅ Answer — Lab Exercise 1 · book documents for semantic search
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
from data import BOOKS
def make_book_document(book):
    # combine title, author, genre, year, description, setting, themes, rating and pages into one text
    document = f"The book titled '{book['title']}' by {book['author']} is a {book['genre']} novel published in {book['year']}. "
    document += f"It tells {book['description']}. "
    document += f"The story is set in {book['setting']} and explores themes such as {book['themes']}. "
    document += f"It has a reader rating of {book['rating']} and spans {book['pages']} pages."
    return document

book_documents = [make_book_document(b) for b in BOOKS]
print(len(book_documents), "documents")
print(book_documents[3])

### Exercise 2 · Implement metadata filtering
Add the book data to a collection with comprehensive metadata for filtering capabilities.

*Hint: include all book attributes as metadata for flexible filtering.*

In [ ]:
# Your code here

In [ ]:
# @title ✅ Answer — Lab Exercise 2 · add the books with full metadata
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
import chromadb
from data import BOOKS
from gemini_ef import GeminiEmbeddingFunction
book_documents = [make_book_document(b) for b in BOOKS]          # from Exercise 1
books_client = chromadb.Client()
books_col = books_client.get_or_create_collection(
    name="books_nb",                                               # own name: does not clash with other collections
    metadata={"description": "A collection for storing books data"},
    configuration={"hnsw": {"space": "cosine"}, "embedding_function": GeminiEmbeddingFunction()})
books_col.upsert(                                                  # upsert = safe to re-run
    ids=[b["id"] for b in BOOKS],
    documents=book_documents,
    metadatas=[{k: b[k] for k in ["title", "author", "genre", "year", "rating", "pages",
                                  "description", "themes", "setting"]} for b in BOOKS])
print(books_col.count(), "books |", books_col.get(ids=["book_7"])["metadatas"][0])

### Exercise 3 · Create an advanced search function
Implement `perform_advanced_search(collection, all_items)` that demonstrates:
- similarity search for *"magical fantasy adventure"*
- filter books by genre (**Fantasy or Science Fiction**)
- filter books by rating (4.0 or higher)
- combined search: find highly-rated dystopian books with similarity search

*Hint: use the same pattern as the employee search but adapt it for book attributes.*

In [ ]:
# Your code here

In [ ]:
# @title ✅ Answer — Lab Exercise 3 · advanced search function
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
def show(results, query=None):
    """Print query() results (nested lists) or get() results (flat lists)."""
    if query is not None:
        for i, (m, d) in enumerate(zip(results["metadatas"][0], results["distances"][0]), 1):
            print(f"  {i}. {m['title']} — {m['author']} ({m['genre']}, {m['year']}, ★{m['rating']})  distance {d:.4f}")
    else:
        for m in results["metadatas"]:
            print(f"  - {m['title']} — {m['author']} ({m['genre']}, ★{m['rating']})")

def perform_advanced_search(collection, all_items):
    print("\n1. Similarity search: 'magical fantasy adventure'")
    show(collection.query(query_texts=["magical fantasy adventure"], n_results=3), query=True)

    print("\n2. Books with genre Fantasy or Science Fiction:")
    r = collection.get(where={"genre": {"$in": ["Fantasy", "Science Fiction"]}})
    print(f"Found {len(r['ids'])} books"); show(r)

    print("\n3. Books with rating 4.0 or higher:")
    r = collection.get(where={"rating": {"$gte": 4.0}})
    print(f"Found {len(r['ids'])} books"); show(r)

    print("\n4. Combined: highly-rated dystopian books (similarity + metadata filter)")
    show(collection.query(query_texts=["dystopian society under oppressive control"], n_results=3,
                          where={"$and": [{"genre": "Dystopian"}, {"rating": {"$gte": 4.0}}]}), query=True)

perform_advanced_search(books_col, books_col.get())

### Exercise 4 · Run your advanced book search
Complete `books_advanced_search.py` below (it starts as a runnable skeleton), then execute it and observe the different types of search results.

*Hint: use the same command pattern as the main lab.*

In [ ]:
%%writefile books_advanced_search.py
# Practice exercise — your books_advanced_search.py (Exercise 4 runs it). Fill in the TODOs.
import chromadb
from gemini_ef import GeminiEmbeddingFunction  # Colab: Gemini embeddings
from data import BOOKS

def perform_advanced_search(collection, all_items):
    # TODO Exercise 3:
    #  1. similarity search for "magical fantasy adventure"
    #  2. filter books by genre (Fantasy or Science Fiction)
    #  3. filter books by rating (4.0 or higher)
    #  4. combined search: highly-rated dystopian books with similarity search
    pass

if __name__ == "__main__":
    ef = GeminiEmbeddingFunction()
    client = chromadb.Client()
    collection = client.create_collection(
        name="books_collection",
        metadata={"description": "A collection for storing books data"},
        configuration={"hnsw": {"space": "cosine"}, "embedding_function": ef})
    book_documents = []
    for book in BOOKS:
        # TODO Exercise 1: build one descriptive text per book (title, description, themes, setting, ...)
        document = book["title"]
        book_documents.append(document)
    # TODO Exercise 2: add the books with all attributes as metadata
    collection.add(ids=[book["id"] for book in BOOKS], documents=book_documents)
    print(f"Collection created: {collection.name}")
    all_items = collection.get()
    print(f"Number of documents: {len(all_items['documents'])}")
    perform_advanced_search(collection, all_items)

In [ ]:
!python books_advanced_search.py

In [ ]:
# @title ✅ Answer — Lab Exercise 4 · run the complete books_advanced_search.py
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
from pathlib import Path
Path("books_advanced_search.py").write_text(r"""
import chromadb
from gemini_ef import GeminiEmbeddingFunction  # Colab: Gemini embeddings (was SentenceTransformerEmbeddingFunction)
from data import BOOKS

def print_query(results):
    for i, (doc_id, document, distance) in enumerate(zip(
            results['ids'][0], results['documents'][0], results['distances'][0])):
        m = results['metadatas'][0][i]
        print(f"  {i+1}. {m['title']} ({doc_id}) - Distance: {distance:.4f}")
        print(f"     author: {m['author']}, genre: {m['genre']}, year: {m['year']}, rating: {m['rating']}")
        print(f"     Document: {document[:100]}...")

def print_get(results):
    for m in results['metadatas']:
        print(f"  - {m['title']}: {m['author']} ({m['genre']}, {m['year']}, rating {m['rating']}, {m['pages']} pages)")

def perform_advanced_search(collection, all_items):
    print("=== Similarity Search ===")
    print("\n1. Searching for magical fantasy adventures:")
    query_text = "magical fantasy adventure"
    print(f"Query: '{query_text}'")
    print_query(collection.query(query_texts=[query_text], n_results=3))

    print("\n=== Metadata Filtering ===")
    print("\n2. Finding all Fantasy or Science Fiction books:")
    results = collection.get(where={"genre": {"$in": ["Fantasy", "Science Fiction"]}})
    print(f"Found {len(results['ids'])} Fantasy / Science Fiction books:")
    print_get(results)

    print("\n3. Finding books with rating 4.0 or higher:")
    results = collection.get(where={"rating": {"$gte": 4.0}})
    print(f"Found {len(results['ids'])} highly rated books:")
    print_get(results)

    print("\n=== Combined Search: Similarity + Metadata Filtering ===")
    print("\n4. Finding highly-rated dystopian books:")
    query_text = "dystopian society under oppressive control"
    print(f"Query: '{query_text}' with filters (genre = Dystopian, rating >= 4.0)")
    print_query(collection.query(query_texts=[query_text], n_results=3,
                                 where={"$and": [{"genre": "Dystopian"}, {"rating": {"$gte": 4.0}}]}))

if __name__ == "__main__":
    ef = GeminiEmbeddingFunction()
    client = chromadb.Client()
    collection = client.create_collection(
        name="books_collection",
        metadata={"description": "A collection for storing books data"},
        configuration={"hnsw": {"space": "cosine"}, "embedding_function": ef})
    book_documents = []
    for book in BOOKS:                                   # Exercise 1
        document = f"The book titled '{book['title']}' by {book['author']} is a {book['genre']} novel published in {book['year']}. "
        document += f"It tells {book['description']}. "
        document += f"The story is set in {book['setting']} and explores themes such as {book['themes']}. "
        document += f"It has a reader rating of {book['rating']} and spans {book['pages']} pages."
        book_documents.append(document)
    collection.add(                                      # Exercise 2
        ids=[book["id"] for book in BOOKS],
        documents=book_documents,
        metadatas=[{k: book[k] for k in ["title", "author", "genre", "year", "rating", "pages",
                                         "description", "themes", "setting"]} for book in BOOKS])
    print(f"Collection created: {collection.name}")
    all_items = collection.get()
    print(f"Number of documents: {len(all_items['documents'])}")
    perform_advanced_search(collection, all_items)       # Exercise 3
""".lstrip())
!python books_advanced_search.py

### Bonus challenge
Extend the search to find books by specific criteria such as:
- books published in a specific decade
- books with similar page counts
- books that match multiple themes

In [ ]:
# Your code here

In [ ]:
# @title ✅ Answer — Lab Bonus Challenge · decade, similar page counts, multiple themes
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
def books_in_decade(collection, decade):
    # a decade = [decade, decade + 9]; e.g. 1950 → 1950–1959
    r = collection.get(where={"$and": [{"year": {"$gte": decade}}, {"year": {"$lte": decade + 9}}]})
    return sorted((m["year"], m["title"]) for m in r["metadatas"])

def books_with_similar_pages(collection, title, tolerance=0.25):
    ref = collection.get(where={"title": title})["metadatas"][0]["pages"]
    lo, hi = int(ref * (1 - tolerance)), int(ref * (1 + tolerance))
    r = collection.get(where={"$and": [{"pages": {"$gte": lo}}, {"pages": {"$lte": hi}}, {"title": {"$ne": title}}]})
    return ref, (lo, hi), sorted((m["pages"], m["title"]) for m in r["metadatas"])

def books_matching_themes(collection, themes, n=3):
    # semantic: embed all themes in one query text, then rank
    r = collection.query(query_texts=[", ".join(themes)], n_results=n)
    semantic = [m["title"] for m in r["metadatas"][0]]
    # exact: every theme word must appear in the document (where_document is case-sensitive!)
    exact = collection.get(where_document={"$and": [{"$contains": t} for t in themes]})
    return semantic, [m["title"] for m in exact["metadatas"]]

print("1950s:", books_in_decade(books_col, 1950))
print("1960s:", books_in_decade(books_col, 1960))
print("pages like 'The Hunger Games':", books_with_similar_pages(books_col, "The Hunger Games"))
print("themes friendship + good vs evil:", books_matching_themes(books_col, ["friendship", "good vs evil"]))

## 🧪 Colab exercises
### C1 · Load the employee collection in the notebook
Import the lab's `EMPLOYEES` from `data.py` and build the same collection in memory so you can query it interactively.

In [ ]:
import chromadb, importlib, data
importlib.reload(data); from data import EMPLOYEES
from chromadb.utils import embedding_functions
from gemini_ef import GeminiEmbeddingFunction  # Colab: Gemini embeddings
ef = GeminiEmbeddingFunction()
client = chromadb.Client()
col = client.get_or_create_collection("employees_nb", configuration={"hnsw": {"space": "cosine"}, "embedding_function": ef})
docs = [f"{e['role']} with {e['experience']} years of experience in {e['department']}. Skills: {e['skills']}. "
        f"Located in {e['location']}. Employment type: {e['employment_type']}." for e in EMPLOYEES]
col.upsert(ids=[e["id"] for e in EMPLOYEES], documents=docs,
           metadatas=[{k: e[k] for k in ["name", "department", "role", "experience", "location", "employment_type"]} for e in EMPLOYEES])
print(col.count(), "employees")

### C2 · Your own filters
Find data/ML people with < 8 years of experience who work full-time; then try `where_document={"$contains": "Python"}`.

In [ ]:
# Your code here
# data/ML people, < 8 years, full-time; then where_document={"$contains": "Python"}

In [ ]:
# @title ✅ Answer — C2 · your own filters
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
r = col.query(query_texts=["machine learning and data analysis"], n_results=5,
              where={"$and": [{"experience": {"$lt": 8}}, {"employment_type": "Full-time"}]})
print("data/ML, < 8 years, full-time:")
for m, d in zip(r["metadatas"][0], r["distances"][0]):
    print(f"  {m['name']:<18} {m['role']:<28} {m['experience']}y  {m['employment_type']}  {d:.3f}")
hits = col.get(where_document={"$contains": "Python"})
print("documents containing 'Python':", [m["name"] for m in hits["metadatas"]])

### C3 · LLM staffing assistant (mini RAG)
Retrieve candidates, then ask `llm` to pick a team and justify it from the records only.

In [ ]:
# Your code here

In [ ]:
# @title ✅ Answer — C3 · LLM staffing assistant (mini RAG)
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
need = "We need two people to build a customer-facing web app with a machine learning recommendation feature."
r = col.query(query_texts=[need], n_results=5)
context = "\n".join(f"- {m['name']}: {d}" for m, d in zip(r["metadatas"][0], r["documents"][0]))
print("Retrieved candidates:\n" + context + "\n")
prompt = (f"Employee records:\n{context}\n\nTask: {need}\n"
          "Pick the best two people using only these records and explain why in 3 bullets. "
          "If the records do not support a choice, say so.")
print(llm.invoke(prompt).text)     # .text: Gemini 3 returns a list of content blocks; .text joins the text parts

### C4 · Try it yourself
#### C4.1
Add a `skills` field to the metadata and filter with `where_document={"$contains": ...}` vs. a metadata field — which is more robust?

In [ ]:
# Your code here

In [ ]:
# @title ✅ Answer — C4.1 · skills: where_document vs metadata
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
for e in EMPLOYEES:   # store skills as a list -> metadata arrays support $contains (exact element match)
    col.update(ids=[e["id"]], metadatas=[{"skills_list": [s.strip() for s in e["skills"].split(",")]}])
by_doc   = {m["name"] for m in col.get(where_document={"$contains": "Python"})["metadatas"]}
by_doc_l = {m["name"] for m in col.get(where_document={"$contains": "python"})["metadatas"]}
by_meta  = {m["name"] for m in col.get(where={"skills_list": {"$contains": "Python"}})["metadatas"]}
print("where_document 'Python':", sorted(by_doc))
print("where_document 'python':", sorted(by_doc_l), "  <- case-sensitive substring")
print("metadata skills_list ∋ 'Python':", sorted(by_meta))
print("$contains on a plain string field:", col.get(where={"name": {"$contains": "John"}})["ids"])

#### C4.2
Swap `GeminiEmbeddingFunction()` for `GeminiEmbeddingFunction("models/gemini-embedding-001")` and compare the top-3 for *"team leader manager with experience"*.

In [ ]:
# Your code here

In [ ]:
# @title ✅ Answer — C4.2 · gemini-embedding-001 vs gemini-embedding-2
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
ef_001 = GeminiEmbeddingFunction("models/gemini-embedding-001")
col_001 = client.get_or_create_collection("employees_nb_001",
                                          configuration={"hnsw": {"space": "cosine"}, "embedding_function": ef_001})
col_001.upsert(ids=[e["id"] for e in EMPLOYEES], documents=docs)
q = "team leader manager with experience"
for label, c in [("embedding-2  ", col), ("embedding-001", col_001)]:
    r = c.query(query_texts=[q], n_results=3)
    print(label, [(EMPLOYEES[int(i.split('_')[1]) - 1]["name"], round(d, 3)) for i, d in zip(r["ids"][0], r["distances"][0])])

#### C4.3
In `books_advanced_search.py`, add a search for *"dystopian future with a strong heroine"* filtered to `rating >= 4.0` (try it in the notebook on `books_col` first).

In [ ]:
# Your code here

In [ ]:
# @title ✅ Answer — C4.3 · dystopian heroine, rating ≥ 4.0
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
r = books_col.query(query_texts=["dystopian future with a strong heroine"], n_results=3,
                    where={"rating": {"$gte": 4.0}})
for m, d in zip(r["metadatas"][0], r["distances"][0]):
    print(f"{m['title']:<42} {m['genre']:<16} ★{m['rating']}  {d:.4f}")
# In books_advanced_search.py add the same call as a 5th block inside perform_advanced_search():
#     print("\n5. Dystopian future with a strong heroine (rating >= 4.0):")
#     print_query(collection.query(query_texts=["dystopian future with a strong heroine"],
#                                  n_results=3, where={"rating": {"$gte": 4.0}}))